# Task 5 ? Latency, drift, lineage and the people

This written analysis uses the PakFreight Task 4 pipeline and its validated figure: **578,650 kg across 34 shipments**. The pipeline has three hidden staging queries and one loaded `Shipments` table.

## 1. Data latency

| Consumer | Required freshness | Justification |
|---|---|---|
| Dispatcher rerouting a truck around a blocked motorway | **Near-real-time** | The decision affects an active trip; a delay of minutes can send the driver into the closure. |
| CFO monthly cost-per-tonne report | **Daily batch** | Finance needs complete, reconciled costs and weights, not second-by-second movement; a daily close supports month-end controls. |
| Customer tracking a consignment online | **Near-real-time** | The customer needs recent milestone/status changes, but a few minutes of source and integration latency is acceptable. |
| Board quarterly mode-profitability review | **Weekly** | The board needs stable, reconciled aggregates; quarterly decisions do not require operational freshness. |

?Always real-time? is wrong because it imposes concrete infrastructure and operating costs: continuous ingestion/stream-processing and high-availability storage are more expensive than scheduled batch processing, and real-time pipelines require more monitoring, on-call support, and failure-recovery complexity. It can also publish incomplete or unvalidated records before duplicate, unit, and reconciliation checks finish, reducing trust in executive figures.

## 2. Schema drift

### New `Wagon_Type` column

The Task 4 `rail_stg` query explicitly selects the eight contract columns with `Table.SelectColumns`. The new source column is therefore **silently dropped** by the current pipeline, while the refresh continues and the eight-column `Shipments` output remains unchanged. Detect it before reporting with a schema contract test that compares the incoming column-name set to the approved rail schema and raises a refresh warning/error for unexpected columns.

### Rename `Freight Charges (PKR)` ? `Charges_PKR`

The current M code references `Freight Charges (PKR)` in `Table.TransformColumnTypes` and later renames it to `CostPKR`; the refresh will **break with a missing-column error**, rather than silently carrying on. Detect it before a report refresh with a required-column test that checks for `Freight Charges (PKR)` and fails the pipeline with a specific drift message if it is absent; alternatively, use a governed source contract/metadata check before Power Query runs.

## 3. Lineage of 578,650 kg

1. **Sources:** `marine_shipments.csv` contributes 12 rows and 206,500 kg; `rail_shipments.xlsx` contributes 10 data rows after its two title rows; `road_shipments.csv` contributes 15 raw lines.
2. **Staging:** Marine dates are parsed as UK `dd/mm/yyyy`; Rail headers are repaired, IDs receive `RL-`, and `Weight (Tons)` is multiplied by 1,000; Road IDs/statuses are standardized.
3. **Row controls:** Road keeps the first `rd-505`, rejects one blank row and rejects the non-data `TOTAL` row, leaving 12 real road shipments.
4. **Conformed storage table:** the three staging contracts append to 34 `Shipments` rows with eight columns and canonical `WeightKG`.
5. **Metric:** `SUM(WeightKG)` = Marine 206,500 + Rail 295,250 + Road 76,900 = **578,650 kg**.
6. **Presentation:** the dashboard card reads the validated sum; any tonnes display divides this canonical kilogram measure by 1,000.

If the CFO says 578,650 kg looks too low, I would check these three places in order: **(1)** the source/staging rows and their row-removal flags, especially the Rail tons-to-kg conversion and Road `TOTAL`/duplicate handling; **(2)** the conformed `Shipments` table, checking 34 rows, source-system subtotals, and `SUM(WeightKG)`; **(3)** the presentation measure/card, confirming it uses the canonical kilogram column once and has not applied an extra filter or unit conversion.

## 4. Roles in the BI ecosystem

| Responsibility | Owner | Main tier |
|---|---|---|
| Designing the star schema for Shipments | **Data architect** | Storage/integrated model |
| Writing and maintaining the Power Query pipeline | **ETL/data engineer** | Staging |
| Agreeing what ?delivered? officially means | **Business/data steward** | Business definition governing staging/storage/presentation |
| Building the dashboard operations opens each morning | **BI developer** | Presentation |
| Making sure the rail export keeps its agreed format | **Source system owner** | Source, before staging |

## 5. Responsibility map

- **Staging ? ETL/data engineer:** ingest raw data, parse dates, standardize names/statuses, convert units, mark or remove invalid rows, and preserve source lineage.
- **Storage ? data architect with the ETL/data engineer:** define the conformed `Shipments` contract/star schema and enforce shared keys, dates, statuses, and `WeightKG`.
- **Presentation ? BI developer, governed by the steward:** publish measures and dashboards from the trusted integrated table.

When nobody owns staging, Operations can include the Road `TOTAL` row, Finance can treat Rail tons as kilograms, and Sales can retain the duplicate?exactly the Task 3 stovepipes. When nobody owns storage conformance, each department can build a separate ?truth?; when nobody owns presentation, users may select a raw/staging table and recreate the wrong total in a visual.

## 6. Character ? discipline

A freight analyst who skips validation turns an apparently precise dashboard into an uncontrolled business decision. If 578,650 kg were wrong and nobody checked, PakFreight could report the wrong cost per tonne, misstate the profitability of a transport mode, and make capacity or pricing decisions from a duplicated shipment, a Rail unit error, or a summary row. Validation is not bureaucracy: the 34-row, 578,650 kg reconciliation is the control that makes the number defensible.

## Investigation order for an unreproducible inherited dashboard

First, establish the **latency expectation and refresh history**: determine whether the dashboard is stale, partially refreshed, or mixing snapshots from different times. Second, trace the **lineage** from the card through its measure, loaded table, staging transformations, and source rows; reproduce the row counts, units, removals, and subtotals. Third, inspect **schema drift** at each refresh boundary?unexpected columns, renamed fields, changed headers, and type/unit changes?because a new column may be silently ignored while a renamed required field may break the query. Finally, rerun the reconciliation gate and only then correct or republish the dashboard.

## Verification

The following cell confirms the known Task 4 control total and source-system subtotals used throughout this analysis.

In [1]:
from pathlib import Path
import pandas as pd
root=Path.cwd()
if not (root/"marine_shipments.csv").exists(): root=Path(r"D:\University\ML Lab\Week02")
marine=pd.read_csv(root/"marine_shipments.csv")
rail=pd.read_excel(root/"rail_shipments.xlsx",skiprows=2)
road=pd.read_csv(root/"road_shipments.csv")
road_kept=road.loc[road.trip_id.notna() & road.trip_id.ne("TOTAL")].drop_duplicates("trip_id")
subtotals={"Marine":float(marine.Weight_KG.sum()),"Rail":float(rail["Weight (Tons)"].sum()*1000),"Road":float(road_kept.weight_kg.sum())}
print(subtotals)
print("Total kg:", sum(subtotals.values()))
assert subtotals == {"Marine":206500.0,"Rail":295250.0,"Road":76900.0}
assert sum(subtotals.values()) == 578650.0

{'Marine': 206500.0, 'Rail': 295250.0, 'Road': 76900.0}
Total kg: 578650.0
